<a href="https://colab.research.google.com/github/aredesog/MTG-Analises/blob/main/NewMTG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Validação externa — MTG-Jamendo

Terceiro dataset (após Spotify e FMA) para testar a mesma tese: rótulos de
gênero em datasets de MIR carregam ambiguidade/arbitrariedade estrutural.

O MTG-Jamendo é MULTI-RÓTULO como o FMA, mas SEM hierarquia declarada
(não tem 'top-level' / árvore de gêneros como o FMA). Por isso a
metodologia usada aqui replica a do SPOTIFY (Seção 4.1-4.2 do paper):
reordenação aleatória + tie-breaking, em vez da comparação de raízes
hierárquicas usada no FMA.

  Q1. Quantas faixas têm MAIS DE UM gênero atribuído? (mesmo teste do FMA)
  Q2. Sob 5 reordenações aleatórias independentes (mesmas seeds do paper:
      42, 7, 13, 99, 123), quantas faixas recebem um gênero diferente
      dependendo só da ordem sorteada? (mesmo teste do Spotify, Seção 4.2)

Fonte: data/autotagging_genre.tsv do repositório oficial MTG/mtg-jamendo-dataset
(https://github.com/MTG/mtg-jamendo-dataset) — 55.215 faixas, já processado
com o tag_map.json oficial deles (não precisamos reaplicar limpeza de tags).
Usamos só o arquivo de metadados (texto), sem baixar áudio nem features.

In [ ]:
# 1 — Carregamento do metadata do MTG-Jamendo
#
# O arquivo é um .tsv de largura VARIÁVEL: as 5 primeiras colunas são fixas
# (TRACK_ID, ARTIST_ID, ALBUM_ID, PATH, DURATION), e cada coluna a partir da
# 6ª é uma tag no formato 'genre---nome'. Uma faixa com 3 gêneros tem 8
# colunas na linha; uma com 1 gênero tem 6. Por isso não dá pra usar
# pd.read_csv direto (colunas ragged) — parseamos manualmente com csv.reader.

import csv
from collections import Counter
import random

path = "autotagging_genre.tsv"  # coloque o arquivo baixado do repo na mesma pasta

tracks = {}
with open(path, newline="", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    header = next(reader)
    for row in reader:
        if not row:
            continue
        track_id = row[0]
        tags_raw = row[5:]
        genres = [t.split("---", 1)[1] for t in tags_raw if t.startswith("genre---")]
        tracks[track_id] = genres

com_genero = {tid: g for tid, g in tracks.items() if len(g) > 0}
print(f"Total de faixas carregadas: {len(tracks):,}")
print(f"Faixas com ao menos um gênero: {len(com_genero):,}")

Total de faixas carregadas: 55,215
Faixas com ao menos um gênero: 55,215


In [ ]:
# 2 — Q1: Ambiguidade multi-rótulo (faixas com >1 gênero)
# Mesmo teste do FMA (Seção 4.6 / Q1 do NewFMA.ipynb), agora no MTG.

multi = {tid: g for tid, g in com_genero.items() if len(g) > 1}

print(f"Q1 — Faixas com MAIS DE UM gênero (multi-rótulo):")
print(f"  {len(multi):,} faixas  ({100*len(multi)/len(com_genero):.1f}% das rotuladas)")

dist = Counter(len(g) for g in com_genero.values())
print(f"\nDistribuição (nº de gêneros por faixa):")
for k in sorted(dist):
    print(f"  {k}: {dist[k]:,}")

Q1 — Faixas com MAIS DE UM gênero (multi-rótulo):
  43,494 faixas  (78.8% das rotuladas)

Distribuição (nº de gêneros por faixa):
  1: 11,721
  2: 16,388
  3: 20,087
  4: 4,704
  5: 1,798
  6: 390
  7: 110
  8: 7
  9: 10


### Por que não usamos 'primeira tag vs última tag da lista'

Uma primeira tentativa foi comparar a tag na primeira posição do arquivo
contra a última, como um proxy rápido de 'arbitrariedade'. O resultado deu
**100% de mudança** — mas isso é um artefato, não um achado: como o MTG não
tem hierarquia (diferente do FMA), duas posições fixas de uma lista com
itens diferentes são quase sempre diferentes por definição. Isso mediria
apenas 'a lista tem mais de um item', não arbitrariedade de verdade.

Por isso a Seção 3 abaixo usa a mesma técnica do **Spotify** (paper,
Seção 4.2): reordenar aleatoriamente sob 5 seeds independentes e medir
quantas faixas mudam de gênero *só por causa da ordem sorteada*.

In [ ]:
# 3 — Q2: Estabilidade sob reordenação aleatória (mesma lógica do Spotify)
#
# Para cada faixa, embaralhamos a lista de gêneros sob 5 seeds independentes
# (as MESMAS seeds usadas no paper original: 42, 7, 13, 99, 123) e tomamos o
# primeiro gênero da lista embaralhada como o 'rótulo único' daquela rodada —
# análogo à desduplicação sob ordenação aleatória feita no Spotify.
#
# Uma faixa é ESTÁVEL se o gênero escolhido for o mesmo nas 5 rodadas;
# caso contrário, é INSTÁVEL — o rótulo dela muda por acaso.

seeds = [42, 7, 13, 99, 123]
resultados_por_seed = {}

for s in seeds:
    rng = random.Random(s)
    escolha = {}
    for tid, g in com_genero.items():
        g_shuffled = g.copy()
        rng.shuffle(g_shuffled)
        escolha[tid] = g_shuffled[0]
    resultados_por_seed[s] = escolha

instaveis = []
n_generos_distintos = []
for tid in com_genero:
    escolhas = set(resultados_por_seed[s][tid] for s in seeds)
    if len(escolhas) > 1:
        instaveis.append(tid)
        n_generos_distintos.append(len(escolhas))

print(f"Q2 — Estabilidade sob 5 reordenações aleatórias independentes:")
print(f"  Faixas ESTÁVEIS (mesmo gênero nas 5 seeds):  {len(com_genero)-len(instaveis):,}")
print(f"  Faixas INSTÁVEIS (gênero muda por acaso):    {len(instaveis):,}")
print(f"  ({100*len(instaveis)/len(com_genero):.1f}% do total rotulado)")
print(f"  ({100*len(instaveis)/len(multi):.1f}% das faixas multi-rótulo)")
print(f"  Gêneros distintos por faixa instável (média): {sum(n_generos_distintos)/len(n_generos_distintos):.2f}")
print(f"\n→ Assim como no Spotify, a única coisa que varia entre as rodadas é a")
print(f"  ordem sorteada — o que muda o rótulo é o acaso, não a música.")

Q2 — Estabilidade sob 5 reordenações aleatórias independentes:
  Faixas ESTÁVEIS (mesmo gênero nas 5 seeds):  12,971
  Faixas INSTÁVEIS (gênero muda por acaso):    42,244
  (76.5% do total rotulado)
  (97.1% das faixas multi-rótulo)
  Gêneros distintos por faixa instável (média): 2.49

→ Assim como no Spotify, a única coisa que varia entre as rodadas é a
  ordem sorteada — o que muda o rótulo é o acaso, não a música.


In [ ]:
# 4 — Tabela-resumo comparativa (Spotify x FMA x MTG-Jamendo)

print("="*72)
print("RESUMO COMPARATIVO — arbitrariedade de rótulo de gênero em MIR")
print("="*72)
print(f"{'Dataset':<15}{'Faixas rotuladas':<20}{'Multi-rótulo':<18}{'Instável/arbitrário'}")
print(f"{'Spotify':<15}{'89,741':<20}{'18.2%':<18}{'16.7%'}")
print(f"{'FMA':<15}{'104,343':<20}{'74.4%':<18}{'57.5%'}")
mtg_multi_pct = f"{100*len(multi)/len(com_genero):.1f}%"
mtg_inst_pct = f"{100*len(instaveis)/len(com_genero):.1f}%"
print(f"{'MTG-Jamendo':<15}{len(com_genero):<20,}{mtg_multi_pct:<18}{mtg_inst_pct}")
print("\nNotas:")
print("- Spotify/FMA: números do paper (Seções 4.1, 4.2, 4.6).")
print("- MTG-Jamendo: 'instável' medido por reordenação aleatória (5 seeds),")
print("  mesma técnica do Spotify — mais direto que a comparação de raízes")
print("  hierárquicas usada no FMA, já que o MTG não declara hierarquia.")

RESUMO COMPARATIVO — arbitrariedade de rótulo de gênero em MIR
Dataset        Faixas rotuladas    Multi-rótulo      Instável/arbitrário
Spotify        89,741              18.2%             16.7%
FMA            104,343             74.4%             57.5%
MTG-Jamendo    55,215              78.8%             76.5%

Notas:
- Spotify/FMA: números do paper (Seções 4.1, 4.2, 4.6).
- MTG-Jamendo: 'instável' medido por reordenação aleatória (5 seeds),
  mesma técnica do Spotify — mais direto que a comparação de raízes
  hierárquicas usada no FMA, já que o MTG não declara hierarquia.


noções futuras: entender como ta sendo feito essa classificação de genro